3.Model Training

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, hour, month, dayofweek, when

spark = SparkSession.builder.getOrCreate()

spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA crime_data")

crime_df = spark.table("crime_parquet_table")

# Feature engineering again (short version)
crime_df = crime_df.withColumn(
    "Date",
    to_timestamp(col("Date"), "MM/dd/yyyy hh:mm:ss a")
)

crime_df = crime_df \
    .withColumn("Hour", hour(col("Date"))) \
    .withColumn("Month", month(col("Date"))) \
    .withColumn("DayOfWeek", dayofweek(col("Date"))) \
    .withColumn("IsWeekend", (dayofweek(col("Date")).isin([1,7])).cast("integer")) \
    .withColumn("label", when(col("Arrest") == True, 1).otherwise(0))

selected_columns = [
    "Year","Month","Hour","DayOfWeek",
    "District","Beat","Community_Area",
    "Primary_Type","Domestic","IsWeekend","label"
]

crime_ml_df = crime_df.select(selected_columns).dropna()

train_df = crime_ml_df.filter(col("Year") < 2021)
val_df   = crime_ml_df.filter((col("Year") >= 2021) & (col("Year") < 2022))
test_df  = crime_ml_df.filter(col("Year") >= 2022)

print("Train:", train_df.count())
print("Validation:", val_df.count())
print("Test:", test_df.count())

Train: 6653918
Validation: 209613
Test: 1024954


In [0]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

primary_indexer = StringIndexer(
    inputCol="Primary_Type",
    outputCol="Primary_Type_index",
    handleInvalid="keep"
)

primary_encoder = OneHotEncoder(
    inputCol="Primary_Type_index",
    outputCol="Primary_Type_vec"
)

assembler = VectorAssembler(
    inputCols=[
        "Year","Month","Hour","DayOfWeek",
        "District","Beat","Community_Area",
        "Domestic","IsWeekend","Primary_Type_vec"
    ],
    outputCol="features",
    handleInvalid="keep"
)

In [0]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=20
)

pipeline_lr = Pipeline(stages=[
    primary_indexer,
    primary_encoder,
    assembler,
    lr
])

lr_model = pipeline_lr.fit(train_df)
predictions_lr = lr_model.transform(test_df)

evaluator = BinaryClassificationEvaluator(labelCol="label")
auc_lr = evaluator.evaluate(predictions_lr)

print("Logistic Regression AUC:", auc_lr)

Logistic Regression AUC: 0.781963847672617


In [0]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=50,
    maxDepth=10
)

pipeline_rf = Pipeline(stages=[
    primary_indexer,
    primary_encoder,
    assembler,
    rf
])

rf_model = pipeline_rf.fit(train_df)
predictions_rf = rf_model.transform(test_df)

auc_rf = evaluator.evaluate(predictions_rf)
print("Random Forest AUC:", auc_rf)

Random Forest AUC: 0.7817712680403641


In [0]:
from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label",
    maxDepth=10
)

pipeline_dt = Pipeline(stages=[
    primary_indexer,
    primary_encoder,
    assembler,
    dt
])

dt_model = pipeline_dt.fit(train_df)
predictions_dt = dt_model.transform(test_df)

auc_dt = evaluator.evaluate(predictions_dt)
print("Decision Tree AUC:", auc_dt)

Decision Tree AUC: 0.2653183136704874


In [0]:
print("Model Comparison (AUC)")
print("------------------------")
print("Logistic Regression:", auc_lr)
print("Random Forest:", auc_rf)
print("Decision Tree:", auc_dt)

Model Comparison (AUC)
------------------------
Logistic Regression: 0.781963847672617
Random Forest: 0.7817712680403641
Decision Tree: 0.2653183136704874


In [0]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

paramGrid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [20, 50]) \
    .addGrid(rf.maxDepth, [5, 10]) \
    .build()

crossval = CrossValidator(
    estimator=pipeline_rf,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=2
)

import os
os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/crime_data/ml_temp"

cv_model = crossval.fit(train_df)

cv_predictions = cv_model.transform(test_df)
auc_cv = evaluator.evaluate(cv_predictions)

print("Tuned Random Forest AUC:", auc_cv)

Tuned Random Forest AUC: 0.7817784411525494


In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.crime_data.ml_models")

cv_model.bestModel.write().overwrite().save(
    "/Volumes/workspace/crime_data/ml_models/best_rf_model"
)

print("Best model saved successfully.")

Best model saved successfully.
